### Phase 3: Deep Learning Multi-Horizon Continuous Glucose Trajectory Forecasting

This notebook trains and evaluates Deep Learning models (**LSTM**, **GRU**, and **MLP Baseline**) to forecast the **full 120-minute post-meal continuous glucose trajectory** (sampled at 5-minute intervals = 24 forecast horizon steps).

#### Objectives:
1. **Multi-Channel Sequence Formatting**: Pair 60 minutes of pre-meal sequence history (CGM, Heart Rate, METs) with static clinical and gut microbiome embeddings.
2. **Group K-Fold Validation**: Evaluate model generalization on unseen participants using 5-fold Group K-Fold CV.
3. **Multi-Horizon Metric Tracking**: Calculate trajectory RMSE and MAE across forecasting horizons ($t+15m, t+30m, t+60m, t+90m, t+120m$).
4. **Trajectory Curve Visualization**: Plot predicted postprandial glucose curves against continuous sensor readings for unseen subjects.

#### 1. Setup & Environment Configurations

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# Add src to path
sys.path.append(os.path.abspath('..'))
from src.dataset_dl import load_and_preprocess_sequences
from src.models_dl import MLPForecaster, LSTMForecaster, GRUForecaster

plt.style.use('seaborn-v0_8-whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")

#### 2. Sequence Extraction & Dataset Formatting

In [ ]:
X_seq, X_stat, Y_traj, metadata, static_cols = load_and_preprocess_sequences(
    master_meals_path='../data/processed/master_meals_features.csv',
    processed_dir='../data/processed',
    pre_window_mins=60,
    post_window_mins=120,
    downsample_factor=5
)

print(f"Loaded {len(X_seq)} sequences.")
print(f"Pre-meal dynamic sequence (60m): {X_seq.shape}")
print(f"Static clinical & microbiome vector: {X_stat.shape}")
print(f"Target continuous trajectory (120m / 5m step = 24 points): {Y_traj.shape}")

#### 3. PyTorch Dataset & Group K-Fold Framework

In [ ]:
class GlucoseTrajectoryDataset(Dataset):
    def __init__(self, x_seq, x_stat, y_traj):
        self.x_seq = torch.tensor(x_seq, dtype=torch.float32)
        self.x_stat = torch.tensor(x_stat, dtype=torch.float32)
        self.y_traj = torch.tensor(y_traj, dtype=torch.float32)
        
    def __len__(self):
        return len(self.x_seq)
        
    def __getitem__(self, idx):
        return self.x_seq[idx], self.x_stat[idx], self.y_traj[idx]

def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x_seq, x_stat, y_traj in dataloader:
        x_seq, x_stat, y_traj = x_seq.to(device), x_stat.to(device), y_traj.to(device)
        optimizer.zero_grad()
        pred = model(x_seq, x_stat)
        loss = criterion(pred, y_traj)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_traj)
    return total_loss / len(dataloader.dataset)

def evaluate_model(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for x_seq, x_stat, y_traj in dataloader:
            x_seq, x_stat, y_traj = x_seq.to(device), x_stat.to(device), y_traj.to(device)
            pred = model(x_seq, x_stat)
            loss = criterion(pred, y_traj)
            total_loss += loss.item() * len(y_traj)
            all_preds.append(pred.cpu().numpy())
            all_targets.append(y_traj.cpu().numpy())
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    return total_loss / len(dataloader.dataset), all_preds, all_targets

#### 4. Train & Cross-Validate Deep Learning Models

In [ ]:
def cross_validate_dl_model(model_cls, X_seq, X_stat, Y_traj, groups, epochs=50, lr=0.001, batch_size=32):
    gkf = GroupKFold(n_splits=5)
    fold_results = []
    all_cv_preds = np.zeros_like(Y_traj)
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_seq, Y_traj, groups)):
        # Standardize sequence channels (GL, HR, METs)
        scaler_seq = StandardScaler()
        N_tr, L, C = X_seq[train_idx].shape
        X_seq_tr_scaled = scaler_seq.fit_transform(X_seq[train_idx].reshape(-1, C)).reshape(N_tr, L, C)
        
        N_val = len(val_idx)
        X_seq_val_scaled = scaler_seq.transform(X_seq[val_idx].reshape(-1, C)).reshape(N_val, L, C)
        
        # Standardize static features
        scaler_stat = StandardScaler()
        X_stat_tr_scaled = scaler_stat.fit_transform(X_stat[train_idx])
        X_stat_val_scaled = scaler_stat.transform(X_stat[val_idx])
        
        # DataLoaders
        train_ds = GlucoseTrajectoryDataset(X_seq_tr_scaled, X_stat_tr_scaled, Y_traj[train_idx])
        val_ds = GlucoseTrajectoryDataset(X_seq_val_scaled, X_stat_val_scaled, Y_traj[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
        
        # Instantiate model
        static_dim = X_stat.shape[1]
        model = model_cls(in_channels=3, static_dim=static_dim, horizon_steps=24).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.HuberLoss()
        
        best_val_loss = float('inf')
        best_preds = None
        
        for epoch in range(epochs):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss, val_preds, val_targets = evaluate_model(model, val_loader, criterion)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_preds = val_preds
                
        all_cv_preds[val_idx] = best_preds
        
        # Metrics
        mae = np.mean(np.abs(best_preds - Y_traj[val_idx]))
        rmse = np.sqrt(np.mean((best_preds - Y_traj[val_idx]) ** 2))
        fold_results.append({'fold': fold + 1, 'mae': mae, 'rmse': rmse})
        print(f"Fold {fold + 1} | Val MAE: {mae:.2f} mg/dL | Val RMSE: {rmse:.2f} mg/dL")
        
    overall_mae = np.mean([r['mae'] for r in fold_results])
    overall_rmse = np.mean([r['rmse'] for r in fold_results])
    print(f"\nOverall CV Performance -> MAE: {overall_mae:.2f} mg/dL | RMSE: {overall_rmse:.2f} mg/dL")
    return overall_mae, overall_rmse, all_cv_preds

#### 5. Evaluate MLP vs. LSTM vs. GRU Architectures

In [ ]:
groups = metadata['subject'].values

print("=== 1. Evaluating MLP Forecaster Baseline ===")
mlp_mae, mlp_rmse, mlp_preds = cross_validate_dl_model(MLPForecaster, X_seq, X_stat, Y_traj, groups, epochs=60)

print("\n=== 2. Evaluating LSTM Forecaster ===")
lstm_mae, lstm_rmse, lstm_preds = cross_validate_dl_model(LSTMForecaster, X_seq, X_stat, Y_traj, groups, epochs=60)

print("\n=== 3. Evaluating GRU Forecaster ===")
gru_mae, gru_rmse, gru_preds = cross_validate_dl_model(GRUForecaster, X_seq, X_stat, Y_traj, groups, epochs=60)

#### 6. Multi-Horizon Metric Breakdown & Trajectory Visualization

In [ ]:
horizons = [15, 30, 60, 90, 120]
horizon_indices = [(h // 5) - 1 for h in horizons]

print("Multi-Horizon Trajectory MAE (mg/dL) Breakdown:")
print("Horizon (mins) | MLP Baseline | LSTM Model | GRU Model")
print("-" * 55)
for h, idx in zip(horizons, horizon_indices):
    mlp_h_mae = np.mean(np.abs(mlp_preds[:, idx] - Y_traj[:, idx]))
    lstm_h_mae = np.mean(np.abs(lstm_preds[:, idx] - Y_traj[:, idx]))
    gru_h_mae = np.mean(np.abs(gru_preds[:, idx] - Y_traj[:, idx]))
    print(f"{h:14d} | {mlp_h_mae:12.2f} | {lstm_h_mae:10.2f} | {gru_h_mae:9.2f}")

In [ ]:
# Plot 4 sample postprandial glucose trajectories overlaying actual vs. predicted curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
time_steps = np.arange(5, 125, 5)

sample_indices = [10, 45, 120, 250]
for ax, sample_idx in zip(axes.flat, sample_indices):
    actual_curve = Y_traj[sample_idx]
    lstm_curve = lstm_preds[sample_idx]
    gru_curve = gru_preds[sample_idx]
    subj = metadata.loc[sample_idx, 'subject']
    meal_time = metadata.loc[sample_idx, 'timestamp']
    
    ax.plot(time_steps, actual_curve, 'o-', label='Actual CGM Reading', color='black', linewidth=2.5)
    ax.plot(time_steps, lstm_curve, 's--', label='LSTM Forecast', color='#1f77b4', linewidth=2)
    ax.plot(time_steps, gru_curve, '^--', label='GRU Forecast', color='#2ca02c', linewidth=2)
    ax.set_title(f"Subject {subj} | Meal Time: {meal_time}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Post-Meal Time (Minutes)", fontsize=10)
    ax.set_ylabel("Glucose (mg/dL)", fontsize=10)
    ax.legend()
    
plt.tight_layout()
plt.show()